<a href="https://colab.research.google.com/github/nauzleyabedini/Clinical-Note-Evaluation/blob/main/Clinical_Note_Eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# AI-Powered Clinical Documentation Evaluation Framework:

This project presents a robust framework for the comprehensive evaluation of AI-generated clinical notes against established quality, accuracy, and compliance standards. Leveraging advanced Large Language Models (LLMs) and quantitative metrics, this system addresses critical challenges in healthcare AI, such as hallucination detection, clinical accuracy, and impact on clinician workflow.

## Project Overview

The framework is designed to meticulously audit AI-generated clinical documentation through a two-phased evaluation process. It integrates multiple state-of-the-art LLMs (OpenAI, Anthropic, Google Gemini) to simulate real-world auditing scenarios and measure the efficiency gains and potential burdens introduced by AI in clinical settings. This project showcases expertise in:

*   **Multi-LLM Integration & Orchestration:** Seamlessly working with and evaluating outputs from diverse LLM providers, including managing model-specific API nuances.
*   **Custom Prompt Engineering:** Developing structured, domain-specific, and rigorously tested prompts for highly accurate and consistent evaluations, including explicit JSON schema enforcement.
*   **Robust API Handling:** Implementing advanced retry logic with exponential backoff and fine-tuning model parameters (e.g., `max_tokens`, `temperature`) to ensure resilience, reliability, and deterministic output from various APIs.
*   **Quantitative Performance Metrics:** Defining and applying measurable metrics for clinical accuracy, compliance (HCC capture, ICD-10 specificity), safety, and clinician editing burden.
*   **Healthcare AI Application:** Addressing a critical need in clinical AI development for rigorous validation and continuous improvement of documentation systems, directly contributing to safe and effective AI deployment in healthcare.

## Methodology

### Phase 1: AI Note Generation & Pre-Vetting Evaluation (AI Note vs. Original Transcript & Gold Standard Note)

This phase begins with the generation of AI-powered clinical notes, followed by a pre-vetting evaluation. It assesses the initial quality of AI-generated draft notes by comparing them against the original doctor-patient transcripts and a "Gold Standard Note". It simulates an expert Clinical Documentation Integrity (CDI) audit using two distinct LLMs (OpenAI `gpt-4o-mini` and Anthropic `claude-sonnet-5`) acting as independent auditors.

**AI Note Generation Process (Google Gemini):
**
*   **Few-Shot Prompting:** The Gemini API (`gemini-3.6-flash`) is used to generate the initial AI draft notes. The prompt incorporates few-shot examples to guide the model towards the desired output format and content, emphasizing clinical integrity, compliance, and safety. This ensures the generated notes adhere to specific stylistic and structural requirements right from the generation stage.
*   **Post-Processing for Structure:** A custom Python function is applied post-generation to enforce a strict, predefined clinical note structure (e.g., 'CHIEF COMPLAINT', 'HISTORY OF PRESENT ILLNESS', 'REVIEW OF SYSTEMS', 'PHYSICAL EXAM', 'RESULTS', 'ASSESSMENT AND PLAN'). This programmatic step uses regular expressions to parse and reconstruct the note, ensuring all required headings are present and content is correctly organized, even if initial LLM output deviates slightly.

**Modular Evaluation Approach:**

The evaluation of these AI-generated notes is modular, grouping criteria into three core domains:

1.  **Clinical Fact Grounding:** This module rigorously assesses the AI note's fidelity to the original patient-doctor transcript. It measures:
    *   **Transcript Fidelity (0-100):** How accurately the AI note reflects the information explicitly stated or clearly implied in the transcript. Deductions are made for any information in the AI note not present in the transcript.
    *   **Hallucinations (0-100):** Identifies and penalizes information in the AI note that is entirely fabricated or not grounded in the transcript.
    *   **Critical Omissions (0-100):** Detects and penalizes critical clinical information present in the transcript that was omitted from the AI note.

2.  **Compliance & Billing:** This module focuses on the adherence to coding and regulatory standards by comparing the AI Note against both the Gold Standard Note and the Transcript. It evaluates:
    *   **MEAT Criteria (0-100):** For chronic conditions listed in the AI note's Assessment and Plan, it ensures that MEAT (Monitor, Evaluate, Assess, Treat) criteria are sufficiently documented, aligning with the Gold Standard Note.
    *   **ICD-10 Specificity (0-100):** Assesses if any implied or explicit ICD-10 codes in the AI note are specific and accurate, based on information from the transcript and gold standard.
    *   **Clinical Validation (0-100):** Verifies that clinical statements and diagnoses in the AI note are clinically sound and supported by both the transcript and the Gold Standard Note.

3.  **Quality, Style & Safety:** This module examines the overall quality, adherence to style guidelines, and potential safety risks within the AI note, using both the Gold Standard Note and Transcript for context. It covers:
    *   **Note Structure (0-100):** Evaluates if the AI note adheres to the required outpatient note structure, penalizing deviations.
    *   **Readability / Clarity (0-100):** Assesses the overall readability and clarity, deducting points for jargon, ambiguity, or poor flow.
    *   **Safety Risk Tier (0-100):** Identifies and penalizes any potential patient safety risks, internal medical contradictions, or misinterpretations that could lead to adverse events.

**Output:** Strictly structured JSON containing quantifiable scores (0-100) for each domain and key findings, rigorously enforced through prompt engineering and `response_format` settings.


## Key Features & Technical Highlights

*   **Domain-Specific Data Handling:** Utilizes the ACI-BENCH dataset, demonstrating experience with specialized healthcare datasets and understanding of clinical data nuances.
*   **Flexible LLM Integration with Nuance Handling:** Designed to easily incorporate and compare performance across various generative AI models, including explicit handling of Anthropic's `TextBlock` responses and setting `temperature=0.0` for deterministic JSON output.
*   **Strict Structured Output Parsing:** Ensures reliable extraction of complex evaluation metrics from LLM responses via explicit JSON schema enforcement and robust post-processing.
*   **Advanced Error Tolerance & Recovery:** Implements robust error handling, including API-specific retry mechanisms with exponential backoff, and strategies to mitigate `JSONDecodeError` by optimizing `max_tokens` and `temperature` for consistent LLM output.
*   **Quantitative Analysis for Clinical Impact:** Focuses on generating measurable data points to inform AI model improvements and assess clinical impact, providing actionable insights for clinical AI development.

## Technologies Used

*   **Python:** Core programming language.
*   **Pandas:** Data manipulation and analysis.
*   **OpenAI API:** For AI model evaluation (`gpt-4o-mini`).
*   **Anthropic API:** For AI model evaluation (`claude-sonnet-5`).
*   **Google Gemini API:** For generating "gold-standard" clinical notes (`gemini-3.6-flash`).
*   **Levenshtein:** For calculating edit distances in post-vetting evaluation.
*   **Matplotlib & Seaborn:** (Implicit for future visualization of evaluation results).
*   **Google Colaboratory:** Development environment, utilizing cloud resources.

## Conclusion

This project demonstrates a rigorous, quantitative, and clinically informed approach to evaluating AI models in sensitive domains like clinical documentation. It highlights capabilities in advanced natural language processing (NLP), multi-LLM orchestration, prompt engineering for clinical integrity, data-driven evaluation methodologies, and a deep understanding of domain-specific challenges in healthcare AI. These skills are directly transferable and critical for roles at the forefront of clinical AI research and development.

In [18]:
#-------------------------------------------------------------------------------
#                                OVERALL EVALUATION APPROACH
#-------------------------------------------------------------------------------
#                         ┌─────────────────────────────────────────┐
#                         │       DOCTOR - PATIENT DIALOGUE         |
#                         │            ACI-BENCH Dataset            |
#                         └────────────────────┬────────────────────┘
#                                              │
#                                              ▼
#                         ┌─────────────────────────────────────────┐
#                         |       AMBIENT AI NOTE GENERATION        |
#                         │              (Gemini API)               |
#                         └────────────────────┬────────────────────┘
#                                              │
#                                              ▼
#                         ┌─────────────────────────────────────────┐
#                         │    EVALUATION AGAINST TRANSCRIPT &      |
#                         |         GOLD STANDARD NOTES             |
#                         │        (OpenAI and Anthropic APIs)      |
#                         └────────────────────┬────────────────────┘
#                                              │
#                                              ▼
#               ┌──────────────────────────────┼──────────────────────────────┐
#               ▼                              ▼                              ▼
    # ┌──────────────────────────┐   ┌──────────────────────────┐   ┌──────────────────────────┐
    # │  Module 1: Clinical Fact │   │ Module 2: Compliance &   │   │   Module 3: Quality,     │
    # │        Grounding         │   │         Billing          │   │      Style & Harm        │
    # ├──────────────────────────┤   ├──────────────────────────┤   ├──────────────────────────┤
    # │ • Transcript Fidelity    │   │ • MEAT Compliance        │   │ • Structure              │
    # │ • Hallucination Counts   │   │ • ICD-10 Specificity     │   │ • Readability / Clarity  │
    # │ • Critical Omissions     │   │ • Clinical Validation    │   │ • Safety Risk Tier       │
    # └────────────┬─────────────┘   └────────────┬─────────────┘   └────────────┬─────────────┘
    #              │                              │                              │
    #              └──────────────────────────────┼──────────────────────────────┘
    #                                             ▼
    #                                [ Evaluation Aggregator ]
    #                             (Combines JSON outputs into
    #                              a unified CDI score card)

In [19]:
import os
import json
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import Levenshtein
import time
import random

!pip install anthropic
!pip install -U google-generativeai # Install or update the google-generativeai package
from google.colab import userdata
from anthropic import Anthropic
import google.generativeai as genai # Reverted import for Gemini to original style
from openai import OpenAI, RateLimitError, APIError # Import specific OpenAI error types
from anthropic import APIStatusError, OverloadedError # Only import specific Anthropic error types needed
from anthropic.types import TextBlock # Added TextBlock import for handling Anthropic responses
from google.api_core.exceptions import GoogleAPIError # Import GoogleAPIError for Gemini

# Map Colab secrets to system environment variables
os.environ["ANTHROPIC_API_KEY"] = userdata.get('AnthropicClinDocEvalAPIKey')
os.environ["OPENAI_API_KEY"] = userdata.get('OpenAIClinDocEvalAPIKey')
os.environ["GEMINI_API_KEY"] = userdata.get('Gemini_Clin_Doc_Eval_APIKey')

# Configure Gemini API key
genai.configure(api_key=os.environ["GEMINI_API_KEY"]) # Keep global configure

# Initialize the clients
anthropic_client = Anthropic()
openai_client = OpenAI()
gemini_model = genai.GenerativeModel('gemini-3.6-flash') # Removed api_key from constructor, relying on global configure

# Test Anthropic with retry logic
max_retries_anthropic = 5
base_delay_anthropic = 1 # seconds
anthropic_response = None

for i in range(max_retries_anthropic):
    try:
        anthropic_response = anthropic_client.messages.create(
            model="claude-sonnet-5", # Corrected model name based on error suggestion
            max_tokens=1000,
            messages=[{"role": "user", "content": "Say hello!"}]
        )
        print("Anthropic says:", anthropic_response.content[0].text)
        break # If successful, break the loop
    except OverloadedError as e:
        if i < max_retries_anthropic - 1:
            delay = base_delay_anthropic * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"Anthropic API overloaded. Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering OverloadedError: {e}")
            # If you want to raise an error after max retries:
            # raise # Re-raise the exception if max retries are exceeded
            print("Skipping Anthropic API call due to persistent overload.")
            break # Exit loop if Anthropic persistently overloaded
    except APIStatusError as e:
        print(f"Anthropic API error: {e}")
        raise # Re-raise other API errors immediately

# Test OpenAI with retry logic
max_retries_openai = 5
base_delay_openai = 1 # seconds
openai_response = None

for i in range(max_retries_openai):
    try:
        openai_response = openai_client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[{"role": "user", "content": "Say hello!"}]
        )
        print("OpenAI says:", openai_response.choices[0].message.content)
        break # If successful, break the loop
    except (RateLimitError, APIError) as e:
        if i < max_retries_openai - 1:
            delay = base_delay_openai * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"OpenAI API error ({type(e).__name__}). Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering OpenAI API error: {e}")
            raise # Re-raise the exception if max retries are exceeded

# Test Gemini with retry logic
max_retries_gemini = 5
base_delay_gemini = 1 # seconds
gemini_response = None

for i in range(max_retries_gemini):
    try:
        gemini_response = gemini_model.generate_content("Say hello!")
        print("Gemini says:", gemini_response.text)
        break # If successful, break the loop
    except GoogleAPIError as e: # Changed genai.APIError to GoogleAPIError
        if i < max_retries_gemini - 1:
            delay = base_delay_gemini * (2 ** i) + random.uniform(0, 1) # Exponential backoff with jitter
            print(f"Gemini API error ({type(e).__name__}). Retrying in {delay:.2f} seconds...")
            time.sleep(delay)
        else:
            print(f"Max retries reached. Still encountering Gemini API error: {e}")
            raise # Re-raise the exception if max retries are exceeded

Anthropic says: Hello! 👋 How are you doing today? I'm happy to help with whatever you need — whether that's answering questions, brainstorming ideas, writing something, solving a problem, or just chatting. What's on your mind?
OpenAI says: Hello! How can I assist you today?
Gemini says: Hello! How can I help you today?


In [20]:
# Install the python-Levenshtein library
!pip install python-Levenshtein

In [21]:
#-----------------------------
  #SET UP ACI-BENCH DATASET
#-----------------------------

# Load the dataset directly from GitHub
url = "https://raw.githubusercontent.com/microsoft/clinical_visit_note_summarization_corpus/refs/heads/main/data/aci-bench/challenge_data/train.csv"
df = pd.read_csv(url)

# View basic dataset information
print(f"Total notes available: {len(df)}")
print("Columns in dataset:", df.columns.tolist())
print("Transcript source types:", df['dataset'].value_counts())

#Take a small sample (3 notes) to reduce computation costs
sample_df = df.head(3).copy()
print(f"Successfully loaded {len(sample_df)} encounter records.")

# Display the first transcript snippet
print("\n--- SAMPLE TRANSCRIPT SNIPPET (Note #1) ---")
print(sample_df['note'].iloc[0][:300] + "...")


Total notes available: 67
Columns in dataset: ['dataset', 'encounter_id', 'dialogue', 'note']
Transcript source types: dataset
aci           35
virtassist    20
virtscribe    12
Name: count, dtype: int64
Successfully loaded 3 encounter records.

--- SAMPLE TRANSCRIPT SNIPPET (Note #1) ---
CHIEF COMPLAINT

Annual exam.

HISTORY OF PRESENT ILLNESS

Martha Collins is a 50-year-old female with a past medical history significant for congestive heart failure, depression, and hypertension who presents for her annual exam. It has been a year since I last saw the patient.

The patient has bee...


In [34]:
import google.generativeai as genai

#-------------------------------------------------------
# PHASE 1: GENERATE GEMINI INITIAL AI DRAFT NOTES
#-------------------------------------------------------

def gem_ai_note(transcript):
    """
    Uses Gemini to generate an initial AI draft note grounded in the transcript.
    This note is designed to be compliant, accurate, and safe according to outpatient note standards.
    """
    system_prompt = """You are an AI assistant specialized in clinical documentation generation and integrity.
    Your task is to generate a comprehensive, accurate, compliant, and safe outpatient note grounded in the included patient-doctor transcript.
    Ensure the note is compliant with clinical standards, avoids hallucinations and omissions, and strictly grounds all details in the transcript.

### GENERATION REQUIREMENTS:
CLINICAL INTEGRITY:
1.  **Transcript Fidelity:** Include only information explicitly stated or clearly implied by the transcript. Strictly avoid adding external knowledge, fabrications, or interpretations not supported by the dialogue.
2.  **Style and Structure:** Structure the note as an outpatient note, with each section as a new paragraph and the following headings: 'CHIEF COMPLAINT,' 'HISTORY OF PRESENT ILLNESS,' 'REVIEW OF SYSTEMS,' 'PHYSICAL EXAM,' 'RESULTS,' and 'ASSESSMENT AND PLAN.' ENSURE ALL THE FOLLOWING HEADINGS ARE PRESENT IN THE OUTPUT, EVEN IF THE SECTION REMAINS BLANK. Avoid repetition of information in different sections. Be comprehensive and concise. Do not include an official physician sign-off line as this is an initial draft.
3.  **Completeness (as per transcript):** Capture all relevant clinical information discussed in the transcript.
4.  **Outpatient Standard:** Generate a note suitable for an outpatient setting.
COMPLIANCE:
5.  **HCC:** For every chronic condition listed in the Assessment/Plan, document at least one MEAT action is documented:
    Monitor (signs, symptoms, test results).
    Evaluate (test results, medication response).
    Assess (status, progress, stability).
    Treat (medications, therapies, referrals).
6.  **ICD-10 Codes:** Use correct ICD-10 codes based on information provided in the transcript.
SAFETY:
**Risk Assessment:** Cross-check and correct for any mistated facts or medical errors that could lead to patient safety events or patient harms. Flag any concerns for clinician review and mark the item.
Return ONLY the AI-generated draft clinical note, formatted as follows:

### EXAMPLE OUTPUT FORMAT:
CHIEF COMPLAINT
[Patient's chief complaint, e.g., 'Annual physical examination.']

HISTORY OF PRESENT ILLNESS
[Relevant history, e.g., 'Mrs. Smith is a 45-year-old female presenting for her annual physical...']

REVIEW OF SYSTEMS
[Review of systems findings, e.g., 'Constitutional: No fever or chills.']

PHYSICAL EXAM
[Physical exam findings, e.g., 'General: Well-appearing, no acute distress.' Include vital signs if available.]

RESULTS
[Relevant test results, e.g., 'Labs: CBC within normal limits.']

ASSESSMENT AND PLAN
[Assessment and plan, e.g., '1. Hypertension (I10): Continue Lisinopril. Monitor BP.']
"""

    user_prompt = f"""### ENCOUNTER TRANSCRIPT:
{transcript}

Please generate the initial AI draft clinical note now:"""

    # Call Gemini API
    response = gemini_model.generate_content(
        contents=f"{system_prompt}\n\n{user_prompt}"
    )
    return response.text.strip()

In [35]:
gem_ai_notes_generated = []

print("\n--- Generating Gemini AI Draft Notes ---")
for i in range(len(sample_df)):
    transcript = sample_df['dialogue'].iloc[i]

    print(f"Generating AI Draft Note for Sample {i+1}...")
    generated_note = gem_ai_note(transcript)
    gem_ai_notes_generated.append(generated_note)

sample_df['gem_ai_note_generated'] = gem_ai_notes_generated
display(sample_df[['encounter_id', 'dialogue', 'note', 'gem_ai_note']].head())


--- Generating Gemini AI Draft Notes ---
Generating AI Draft Note for Sample 1...
Generating AI Draft Note for Sample 2...
Generating AI Draft Note for Sample 3...


,encounter_id,dialogue,note,gem_ai_note
0,D2N001,"[doctor] hi , martha . how are you ?\n[patient...",CHIEF COMPLAINT\n\nAnnual exam.\n\nHISTORY OF ...,**SUBJECTIVE:**\n* **Reason for Visit:** Annua...
1,D2N002,"[doctor] hi , andrew , how are you ?\n[patient...",CHIEF COMPLAINT\n\nJoint pain.\n\nHISTORY OF P...,**SUBJECTIVE**\n\n**Patient:** Andrew \n**Age...
2,D2N003,"[doctor] hi , john . how are you ?\n[patient] ...",CHIEF COMPLAINT\n\nBack pain.\n\nHISTORY OF PR...,**SUBJECTIVE**\n* **Patient Identification:** ...


In [37]:
import re # Import regular expression module

# POST-PROCESSING TO FORCE NOTE STRUCTURE - did this because sometimes the longer instructions in the system prompt led to a default to SOAP note format without the additional subheadings.

def post_process_note_structure(note_text):
    required_headings = [
        "CHIEF COMPLAINT",
        "HISTORY OF PRESENT ILLNESS",
        "REVIEW OF SYSTEMS",
        "PHYSICAL EXAM",
        "RESULTS",
        "ASSESSMENT AND PLAN"
    ]

    sections = {}
    current_heading = None
    for line in note_text.split('\n'):
        line_stripped = line.strip()

        found_heading = False
        for heading in required_headings:
            # Use regex to match the heading exactly, ignoring case and optional trailing colon/whitespace
            if re.match(r'^\s*' + re.escape(heading) + r':?\s*$', line_stripped, re.IGNORECASE):
                current_heading = heading
                sections[current_heading] = [] # Initialize list for this new section
                found_heading = True
                break
        # If not a heading and a current_heading has been set, append the line (even if empty)
        if not found_heading and current_heading is not None:
            sections[current_heading].append(line)

    # Reconstruct the note, ensuring all headings are present and content is properly formatted
    processed_note_lines = []
    for heading in required_headings:
        processed_note_lines.append(heading)
        if heading in sections and sections[heading]:
            # Filter out lines that are purely whitespace, but keep empty lines for formatting within content
            content_lines = []
            last_line_was_empty = True
            for ln in sections[heading]:
                if ln.strip(): # If line has content
                    content_lines.append(ln.strip())
                    last_line_was_empty = False
                elif not last_line_was_empty: # If line is empty and previous wasn't, add one empty line
                    content_lines.append('')
                    last_line_was_empty = True

            # Remove any trailing empty lines after processing
            while content_lines and not content_lines[-1].strip():
                content_lines.pop()

            if content_lines:
                processed_note_lines.extend(content_lines)
            else:
                processed_note_lines.append("[No information provided]")
        else:
            processed_note_lines.append("[No information provided]")
        processed_note_lines.append('') # Add an empty line for separation between sections

    return '\n'.join(processed_note_lines).strip()

# Apply post-processing to the 'gem_ai_note_generated' column
print("\n--- Applying Post-Processing to Generated Notes ---")
sample_df['gem_ai_note_post_processed'] = sample_df['gem_ai_note_generated'].apply(post_process_note_structure)

# Display the original generated note and the post-processed version for comparison
display(sample_df[['encounter_id', 'gem_ai_note_generated', 'gem_ai_note_post_processed']].head())


--- Applying Post-Processing to Generated Notes ---


,encounter_id,gem_ai_note_generated,gem_ai_note_post_processed
0,D2N001,CHIEF COMPLAINT\nAnnual physical examination.\...,CHIEF COMPLAINT\nAnnual physical examination.\...
1,D2N002,CHIEF COMPLAINT\nBilateral knee pain.\n\nHISTO...,CHIEF COMPLAINT\nBilateral knee pain.\n\nHISTO...
2,D2N003,CHIEF COMPLAINT\nBack pain.\n\nHISTORY OF PRES...,CHIEF COMPLAINT\nBack pain.\n\nHISTORY OF PRES...


In [42]:
#----------------------------------------------------------------------------
#EVAL OF AI GENERATED NOTE AGAINST GROUND TRUTH TRANSCRIPT & NOTE (2 API JUDGES)
#----------------------------------------------------------------------------

# Universal Clinical Audit Prompt Template
AUDIT_PROMPT_TEMPLATE = """You are an expert Clinical Documentation Integrity (CDI) Auditor and Compliance Specialist.
Audit the following outpatient Gemini AI-generated draft note against the provided documents.

### TRANSCRIPT:
{transcript}

### AI DRAFT NOTE:
{gem_ai_note}

### GOLD STANDARD NOTE (for comparison where relevant):
{gold_standard_note}

Perform a modular evaluation on the following three domains:

---
**Module 1: Clinical Fact Grounding** (Compare AI Draft Note against Transcript)
1.  **Transcript Fidelity (0-100):** Assess how accurately the AI note reflects the information in the transcript. Deduct points for any information in the AI note not present in the transcript.
2.  **Hallucinations (0-100):** Identify any information in the AI note that is not grounded in the transcript. Deduct points for each hallucination.
3.  **Critical Omissions (0-100):** Identify any critical clinical information present in the transcript that was omitted from the AI note. Deduct points for each critical omission.

---
**Module 2: Compliance & Billing** (Compare AI Draft Note against Gold Standard Note and Transcript)
4.  **MEAT Criteria (0-100):** For chronic conditions listed in the AI note's Assessment and Plan, compare against the Gold Standard Note to ensure MEAT (Monitor, Evaluate, Assess, Treat) criteria are sufficiently documented. Deduct points if MEAT is not met.
5.  **ICD-10 Specificity (0-100):** Evaluate if ICD-10 codes in the AI note (if present or implied) are specific and accurate according to the information in the transcript and gold standard. Deduct points for lack of specificity or inaccuracy.
6.  **Clinical Validation (0-100):** Assess if the clinical statements and diagnoses in the AI note are clinically sound and supported by both the transcript and the Gold Standard Note. Deduct points for any clinically unsupported statements.

---
**Module 3: Quality, Style & Safety** (Compare AI Draft Note against Gold Standard Note and Transcript for Safety)
7.  **Note Structure (0-100):** Evaluate if the AI note follows the required outpatient note structure (CHIEF COMPLAINT, HISTORY OF PRESENT ILLNESS, etc.). Deduct points for structural deviations.
8.  **Readability / Clarity (0-100):** Assess the overall readability and clarity of the AI note. Deduct points for jargon, ambiguity, or poor flow.
9.  **Safety Risk Tier (0-100):** Identify any potential patient safety risks, internal medical contradictions, or misinterpretations in the AI note, considering both the transcript and gold standard. Deduct points for each safety risk.

STRICT REQUIREMENT: Reply ONLY in valid JSON matching this schema:
```json
{{
  "module_1_clinical_fact_grounding": {{
    "transcript_fidelity_score": <int 0-100>,
    "hallucinations_score": <int 0-100>,
    "critical_omissions_score": <int 0-100>
  }},
  "module_2_compliance_billing": {{
    "meat_criteria_score": <int 0-100>,
    "icd_10_specificity_score": <int 0-100>,
    "clinical_validation_score": <int 0-100>
  }},
  "module_3_quality_style_safety": {{
    "note_structure_score": <int 0-100>,
    "readability_clarity_score": <int 0-100>,
    "safety_risk_tier_score": <int 0-100>
  }},
  "overall_key_findings": "<short summary of overall errors or strengths across all modules>"
}}
```
"""

def evaluate_with_openai(transcript, gem_ainote, gold_standard_note):
    """Judge 1: OpenAI gpt-4o-mini"""
    prompt = AUDIT_PROMPT_TEMPLATE.format(transcript=transcript, gem_ai_note=gem_ainote, gold_standard_note=gold_standard_note)

    response = openai_client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.0,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": prompt}]
    )
    return json.loads(response.choices[0].message.content)

def evaluate_with_anthropic(transcript, gem_ainote, gold_standard_note):
    """Judge 2: Anthropic claude-sonnet-5"""
    prompt = AUDIT_PROMPT_TEMPLATE.format(transcript=transcript, gem_ai_note=gem_ainote, gold_standard_note=gold_standard_note)

    max_retries = 3
    for i in range(max_retries):
        try:
            response = anthropic_client.messages.create(
                model="claude-sonnet-5",
                max_tokens=4096, # Increased max_tokens to allow for full response including thinking blocks
                temperature=0.0, # Set temperature to 0.0 for deterministic JSON output
                messages=[{"role": "user", "content": prompt}]
            )

            raw_text = ""
            # Iterate through the content blocks to find the first TextBlock
            for block in response.content:
                # Check if the block is an instance of TextBlock
                if isinstance(block, TextBlock):
                    raw_text = block.text
                    break # Once a TextBlock is found, use its text and exit the loop

            # If no TextBlock was found in the response, raise an error to trigger retry or final error
            if not raw_text:
                raise ValueError(f"Anthropic response did not contain a TextBlock. Full content: {response.content}")

            clean_json = raw_text.replace("```json", "").replace("```", "").strip()
            return json.loads(clean_json)
        except ValueError as e:
            if i < max_retries - 1:
                print(f"Anthropic TextBlock not found error: {e}. Retrying...")
                time.sleep(2 ** i) # Exponential backoff
            else:
                raise # Re-raise if max retries reached
        except Exception as e: # Catch other potential API errors during call
            print(f"An unexpected error occurred during Anthropic API call: {e}. Retrying...")
            time.sleep(2 ** i)
    return {} # Should not be reached if retries are exhaustive, but good for safety

In [23]:
transcript_to_evaluate = sample_df['dialogue'].iloc[0]
ai_note_to_evaluate = sample_df['gem_ai_note'].iloc[0]

# Evaluate with OpenAI
openai_evaluation_result = evaluate_with_openai(transcript_to_evaluate, ai_note_to_evaluate)
display(openai_evaluation_result)

# Evaluate with Anthropic
anthropic_evaluation_result = evaluate_with_anthropic(transcript_to_evaluate, ai_note_to_evaluate)
display(anthropic_evaluation_result)

{'accuracy_score': 90,
 'hcc_capture_score': 85,
 'safety_score': 95,
 'key_findings': "The note accurately reflects the patient's history and current status, but it omits specific details about the patient's medication adherence and the impact of stress on her hypertension. HCC capture is strong but could benefit from more specificity regarding the management of chronic conditions. No internal contradictions were found."}

{'accuracy_score': 90,
 'hcc_capture_score': 88,
 'safety_score': 80,
 'key_findings': "Overall accurate and well-organized note reflecting the transcript. Notable internal contradiction: CHF section states patient 'has been compliant with her medication,' while the Hypertension section correctly notes she has been 'inconsistent with the use of her medication' (referring to lisinopril, used for both conditions) — this is a safety/consistency concern. Physical exam section omits the doctor's statement that overall exam was normal aside from the murmur and edema (minor omission, not fabrication). HCC capture is strong: CHF documented with EF 45%, mitral regurgitation, medication titration (lisinopril increase, new Lasix); HTN documented with poor control, non-adherence, and titration; Depression documented with therapy compliance and denial of SI/HI — all reflect adequate MEAT criteria. No fabricated findings identified; vitals and labs align with transcript. Minor improvement opportunit

In [43]:
all_eval_results = []

print("\n--- Evaluating first 3 sample notes ---")
for i in range(len(sample_df.head(3))):
    transcript_to_evaluate = sample_df['dialogue'].iloc[i]
    ai_note_to_evaluate = sample_df['gem_ai_note_post_processed'].iloc[i] # Use the post-processed note
    gold_standard_note_for_eval = sample_df['note'].iloc[i] # This is the 'ground truth' note

    print(f"\n--- Sample Note {i+1} ---")

    # Evaluate with OpenAI
    print("OpenAI Evaluation:")
    openai_evaluation_result = evaluate_with_openai(transcript_to_evaluate, ai_note_to_evaluate, gold_standard_note_for_eval)
    openai_evaluation_result['model'] = 'OpenAI'
    openai_evaluation_result['sample_id'] = i + 1
    all_eval_results.append(openai_evaluation_result)
    display(openai_evaluation_result)

    # Evaluate with Anthropic
    print("Anthropic Evaluation:")
    anthropic_evaluation_result = evaluate_with_anthropic(transcript_to_evaluate, ai_note_to_evaluate, gold_standard_note_for_eval)
    anthropic_evaluation_result['model'] = 'Anthropic'
    anthropic_evaluation_result['sample_id'] = i + 1
    all_eval_results.append(anthropic_evaluation_result)
    display(anthropic_evaluation_result)

print("\n--- Aggregating results ---")
# Flatten the nested dictionaries from LLM responses into a single DataFrame
flattened_results = []
for res in all_eval_results:
    flattened_res = {'model': res['model'], 'sample_id': res['sample_id'], 'overall_key_findings': res['overall_key_findings']}
    for module_name, scores_dict in res.items():
        if isinstance(scores_dict, dict):
            for score_name, score_value in scores_dict.items():
                flattened_res[score_name] = score_value
    flattened_results.append(flattened_res)

aggregated_df = pd.DataFrame(flattened_results)
display(aggregated_df)


--- Evaluating first 3 sample notes ---

--- Sample Note 1 ---
OpenAI Evaluation:


{'module_1_clinical_fact_grounding': {'transcript_fidelity_score': 90,
  'hallucinations_score': 100,
  'critical_omissions_score': 80},
 'module_2_compliance_billing': {'meat_criteria_score': 85,
  'icd_10_specificity_score': 90,
  'clinical_validation_score': 90},
 'module_3_quality_style_safety': {'note_structure_score': 95,
  'readability_clarity_score': 90,
  'safety_risk_tier_score': 80},
 'overall_key_findings': "The AI draft note generally reflects the transcript well but contains hallucinations and critical omissions regarding the patient's travel and specific medication adherence issues. The MEAT criteria are mostly met, and ICD-10 codes are appropriate. The note structure is solid, but there are some readability issues and potential safety risks related to medication management.",
 'model': 'OpenAI',
 'sample_id': 1}

Anthropic Evaluation:
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdkdjMYj7ByF4Ar1m8GJ6'}. Retrying...
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdkdjSuvw9nPQ6C6Z97gm'}. Retrying...
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdkdjcDmj2zc38EWDZ3Pg'}. Retrying...


{'model': 'Anthropic', 'sample_id': 1}


--- Sample Note 2 ---
OpenAI Evaluation:


...

{'module_1_clinical_fact_grounding': {'transcript_fidelity_score': 90,
  'hallucinations_score': 100,
  'critical_omissions_score': 80},
 'module_2_compliance_billing': {'meat_criteria_score': 90,
  'icd_10_specificity_score': 90,
  'clinical_validation_score': 90},
 'module_3_quality_style_safety': {'note_structure_score': 95,
  'readability_clarity_score': 85,
  'safety_risk_tier_score': 80},
 'overall_key_findings': 'The AI note generally reflects the transcript well but contains some hallucinations and critical omissions, particularly regarding symptoms and patient history. Compliance with MEAT criteria is mostly met, and ICD-10 codes are appropriate. The note structure is solid, but readability could be improved, and there are safety risks related to medication interactions that need addressing.',
 'model': 'OpenAI',
 'sample_id': 3}

Anthropic Evaluation:
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011Cdkdkx1BNRvWS9iJGSzR4'}. Retrying...
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011Cdkdm2y5i1MJMNABNKDuX'}. Retrying...
An unexpected error occurred during Anthropic API call: Error code: 400 - {'type': 'error', 'error': {'type': 'invalid_request_error', 'message': '`temperature` is deprecated for this model.'}, 'request_id': 'req_011CdkdmCz5kTd9u4Fxx9fVC'}. Retrying...


{'model': 'Anthropic', 'sample_id': 3}


--- Aggregating results ---


KeyError: 'overall_key_findings'

In [25]:
#Compare and contrast major similarities and differences between the evaluations of OpenAI and Anthropic

In [28]:
# #---------------------------------------------------------------------------
# #PHASE 2: POST-VETTING EVAL (Pre-Signed AI Note vs. Final Revised/Signed)
# #WIP
# #---------------------------------------------------------------------------

# def compute_post_vetting_edits(gem_ai_note, gold_note):
#     """
#     Measures how much the synthetic attending physician (Dr. Gemini) had to edit
#     the raw AI draft note before signing it.
#     """
#     raw_distance = Levenshtein.distance(gem_ai_note, gold_note)
#     max_len = max(len(gem_ai_note), len(gold_note))
#     normalized_edit_dist = raw_distance / max_len if max_len > 0 else 0.0

#     draft_words = gem_ai_note.split()
#     signed_words = gold_note.split()
#     word_distance = Levenshtein.distance(draft_words, signed_words)
#     max_words = max(len(draft_words), len(signed_words))
#     word_edit_ratio = word_distance / max_words if max_words > 0 else 0.0

#     return {
#         "character_edit_distance": raw_distance,
#         "normalized_edit_distance": round(normalized_edit_dist, 4),
#         "word_edit_ratio": round(word_edit_ratio, 4)
#     }

In [29]:
# edit_results = []
# # Iterate over sample_df directly as it now contains 'gem_ai_note' and 'note'
# for index, row in sample_df.iterrows():
#     gem_ai_note_for_edit = row['gem_ai_note'] # This is the Gemini-generated AI note
#     gold_note_from_dataset = row['note'] # This is the 'ground truth' note from the dataset
#     edits = compute_post_vetting_edits(gem_ai_note_for_edit, gold_note_from_dataset)
#     # Add an identifier for the sample
#     edits['sample_id'] = row['encounter_id']
#     edit_results.append(edits)

# edits_df = pd.DataFrame(edit_results)
# # Merge edit_results back into sample_df or display independently
# display(edits_df[['sample_id', 'character_edit_distance', 'normalized_edit_distance', 'word_edit_ratio']])

,sample_id,character_edit_distance,normalized_edit_distance,word_edit_ratio
0,1,3225,0.6833,0.9250
1,2,3330,0.6852,0.8939
2,3,3079,0.6829,0.8824
